Import Libraries

In [63]:
import os
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt


 
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

Data Processing

In [64]:
root ="news-dataset/news-dataset"

lang_codes =  {"english": "eng", "isixhosa": "xho", "chishona": "sna"}

def load_language(lang_code, data_root=root):
    #Load train/dev/test tsv files for one lang
 
    splits = {}
    for split in ["train", "dev", "test"]:
        path = os.path.join(data_root, lang_code, f"{split}.tsv")
        df = pd.read_csv(path, sep="\t")
        df["full_text"] = df["headline"].astype(str) + " " + df["text"].astype(str)
#Builds a full text column from headline and text

        splits[split] = df
        
    return splits



def get_label_encoder(train_df, label_col="category"):
    le = LabelEncoder()
    le.fit(train_df[label_col])
    return le



In [65]:
# test for english load lang
eng = load_language("eng")
for split, df in eng.items():
    print(split, df.shape)
eng["train"].head()

train (3309, 5)
dev (472, 5)
test (948, 5)


,category,headline,text,url,full_text
0,business,'We haven't had a single penny from the Post O...,Baljit Sethi cannot understand why it is takin...,/news/business-63889700,'We haven't had a single penny from the Post O...
1,entertainment,Redcar Regent Cinema: New venue to open on Fri...,kets have gone on sale for a new cinema which ...,/news/uk-england-tees-63260831,Redcar Regent Cinema: New venue to open on Fri...
2,health,Dorset County Hospital stands down critical in...,A main hospital has stood down its critical in...,/news/uk-england-dorset-64130718,Dorset County Hospital stands down critical in...
3,health,Watch: On the picket line with nurses across t...,"Nurses in Northern Ireland, Wales and England ...",/news/uk-64041760,Watch: On the picket line with nurses across t...
4,business,Sri Lanka urges farmers to plant more rice ami...,Sri Lanka is calling on farmers to grow more r...,/news/business-61655317,Sri Lanka urges farmers to plant more rice ami...


In [66]:
#test label encoder
eng_le = get_label_encoder(eng["train"])
print(eng_le.classes_)
print(eng_le.transform(["business", "sports", "technology"]))

['business' 'entertainment' 'health' 'politics' 'sports' 'technology']
[0 4 5]


Text cleaning and vectorisation

In [67]:
#Cleaning

import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)   
    # strip punctuation
    text = re.sub(r"\s+", " ", text).strip()  
    # strip whitespace
    return text

def add_clean_column(df):
    df = df.copy()
    df["clean_text"] = df["full_text"].apply(clean_text)
    return df

In [68]:
#Vectorisation 


def build_vectoriser(method="tfidf", max_features=None, min_freq=1):
    if method == "cvec":
        return CountVectorizer(max_features=max_features, min_df=min_freq)
    elif method == "tfidf":
        return TfidfVectorizer(max_features=max_features, min_df=min_freq)
    else:
        raise ValueError(f"Unknown method: {method}")



def vectorise_splits(splits, method="tfidf", max_features=5000, min_freq=1):
    
    for name in splits:
        splits[name] = add_clean_column(splits[name])

    # make vectoriser
    vectoriser = build_vectoriser(method, max_features, min_freq)


    # choose which words make it and which dont
    x_train = vectoriser.fit_transform(splits["train"]["clean_text"])

    x_dev = vectoriser.transform(splits["dev"]["clean_text"])
    x_test = vectoriser.transform(splits["test"]["clean_text"])

    return (x_train, x_dev, x_test), vectoriser


In [69]:
#Tensors and data loaders 

def prepare_tensors(splits, vectoriser_output, label_col = "category"):
    (x_train, x_dev, x_test), vectorizer = vectoriser_output
    le = get_label_encoder(splits["train"], label_col)

    y_train = le.transform(splits["train"][label_col])
    y_dev   = le.transform(splits["dev"][label_col])
    y_test  = le.transform(splits["test"][label_col])

    return {
        "x_train": x_train, "y_train": y_train,
        "x_dev": x_dev,     "y_dev": y_dev,
        "x_test": x_test,   "y_test": y_test,
         "label_encoder": le,
    }


def make_dataloader(x,y, batch_size = 32, shuffle = False):
    x_dense = torch.tensor(x.toarray(), dtype = torch.float32)
    y_tensor = torch.tensor (y, dtype = torch.long)

    dataset = TensorDataset(x_dense, y_tensor)


    return DataLoader(dataset, batch_size = batch_size, shuffle = shuffle)

def build_dataloaders(tensors, batch_size= 32):
    train_loader = make_dataloader(tensors["x_train"], tensors["y_train"], batch_size, shuffle =True)
    dev_loader = make_dataloader(tensors["x_dev"], tensors["y_dev"], batch_size, shuffle =True)
    test_loader = make_dataloader(tensors["x_test"], tensors["y_test"], batch_size, shuffle =True)

    return train_loader, dev_loader, test_loader
    



In [70]:
#test vectorise splits, prep tensor and dataloader

vec_output = vectorise_splits(eng, method="tfidf", max_features=5000, min_freq=1)
tensors = prepare_tensors(eng, vec_output)
train_loader, dev_loader, test_loader = build_dataloaders(tensors, batch_size=32)

x_batch, y_batch = next(iter(train_loader))
print(x_batch.shape, x_batch.dtype)
print(y_batch.shape, y_batch.dtype)

torch.Size([32, 5000]) torch.float32
torch.Size([32]) torch.int64


Multinominal Logistic Regression

In [71]:
class MultinomialLogisticRegression(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.linear = nn.Linear(input_size, num_classes)


    def forward(self,x):
        return self.linear(x)


    def compute_probabilites(self, x):
        logits = self.forward(x)
        return F.softmax(logits, dim=1)

    def predict(self, x):
        prob = self.compute_probabilites(x)
        return torch.argmax(prob, dim= 1)

In [72]:
#testing  MultiNomialLogReg

test_model= MultinomialLogisticRegression(input_size= x_batch.shape[1], num_classes= len(tensors["label_encoder"].classes_))

logits =  test_model (x_batch)
prob = test_model.compute_probabilites(x_batch)
predictions = test_model.predict(x_batch)


print (logits.shape, prob.shape, predictions.shape)
print (prob[0])
print(prob[0].sum())
print(predictions[:5])


torch.Size([32, 6]) torch.Size([32, 6]) torch.Size([32])
tensor([0.1649, 0.1694, 0.1662, 0.1650, 0.1700, 0.1645],
       grad_fn=<SelectBackward0>)
tensor(1., grad_fn=<SumBackward0>)
tensor([4, 5, 1, 1, 1])


Training and early stopping

In [73]:
def train_model (model, train_loader , dev_loader, lr=0.01, num_epochs=20, patience=3):
    optimiser = torch.optim.SGD(model.parameters(), lr=lr)
    Loss = nn.CrossEntropyLoss()


    history = {"train_loss": [], 
               "Dev_loss":[],
               "Dev_acc":[] }

    best_acc =0
    best_model =None
    no_improvement =0

#train
    for epoch in range(num_epochs):

        model.train()
        train_loss= 0

        for x,y in train_loader:
            optimiser.zero_grad()


            output = model(x)
            loss = Loss(output,y)


            loss.backward()
            optimiser.step()


            train_loss = train_loss + loss.item()*x.size(0)
        train_loss = train_loss/ len(train_loader.dataset)

#validate
        
        model.eval()
        dev_loss =0
        correct = 0

        with torch.no_grad():

            for x,y in dev_loader:
                output = model(x)
                loss = Loss(output,y)

                dev_loss = dev_loss + loss.item()*x.size(0)
                correct = correct + (output.argmax(1)==y).sum().item()

        dev_loss = dev_loss / len(dev_loader.dataset)
        dev_acc = correct/ len(dev_loader.dataset)


        history["train_loss"].append(train_loss)
        history["Dev_loss"].append(dev_loss)
     
        history["Dev_acc"].append(dev_acc)


        print (f"Epoch {epoch+1}/{num_epochs}", f"Train loss:{train_loss:.4f}", f"Dev loss: {dev_loss:.4f}",f"Dev Accuracy:{dev_acc:.4f}")


        #early stop

        if  dev_acc>best_acc:
            best_acc= dev_acc
            best_model = model.state_dict().copy()
            no_improvement = 0
        else:
            no_improvement = no_improvement+1

        if no_improvement>= patience:
            print ("early stop")
            break

    if best_model is not None:
        model.load_state_dict(best_model
                              )


    return history









In [74]:
#test train model
test_model = MultinomialLogisticRegression(input_size=x_batch.shape[1], num_classes=len(tensors["label_encoder"].classes_))

history = train_model(test_model, train_loader, dev_loader, lr=0.1, num_epochs=50, patience = 3)

Epoch 1/50 Train loss:1.7645 Dev loss: 1.7365 Dev Accuracy:0.2288
Epoch 2/50 Train loss:1.7166 Dev loss: 1.6923 Dev Accuracy:0.2966
Epoch 3/50 Train loss:1.6733 Dev loss: 1.6504 Dev Accuracy:0.4364
Epoch 4/50 Train loss:1.6316 Dev loss: 1.6102 Dev Accuracy:0.5021
Epoch 5/50 Train loss:1.5916 Dev loss: 1.5716 Dev Accuracy:0.5593
Epoch 6/50 Train loss:1.5537 Dev loss: 1.5345 Dev Accuracy:0.5869
Epoch 7/50 Train loss:1.5171 Dev loss: 1.4989 Dev Accuracy:0.6250
Epoch 8/50 Train loss:1.4817 Dev loss: 1.4650 Dev Accuracy:0.7203
Epoch 9/50 Train loss:1.4483 Dev loss: 1.4325 Dev Accuracy:0.7267
Epoch 10/50 Train loss:1.4161 Dev loss: 1.4015 Dev Accuracy:0.7458
Epoch 11/50 Train loss:1.3854 Dev loss: 1.3717 Dev Accuracy:0.7585
Epoch 12/50 Train loss:1.3559 Dev loss: 1.3436 Dev Accuracy:0.7585
Epoch 13/50 Train loss:1.3276 Dev loss: 1.3162 Dev Accuracy:0.7860
Epoch 14/50 Train loss:1.3005 Dev loss: 1.2900 Dev Accuracy:0.8072
Epoch 15/50 Train loss:1.2744 Dev loss: 1.2653 Dev Accuracy:0.8136
Epoc

In [75]:
#training evaluation

def evaluate(model, loader, label_encoder):
    model.eval()

    predicitons= []
    labels =[]


    with torch.no_grad():
        for x,y in loader:
            predicitons.extend(model.predict(x).tolist())
            labels.extend(y.tolist())


    #acc metrics

    acc = accuracy_score(labels, predicitons)

    micro = precision_recall_fscore_support(labels, predicitons, average ="micro", zero_division=0)

    macro = precision_recall_fscore_support(labels, predicitons, average="macro", zero_division=0)


    results = {
        "accuracy": acc,
        "micro_precision": micro[0],
        "micro_recall": micro[1],
        "micro_f1": micro[2],
        "macro_precision": macro[0],
        "macro_recall": macro[1],
        "macro_f1": macro[2]
    }


    report = classification_report(
    labels,
    predicitons,
    target_names=label_encoder.classes_,
    zero_division=0
    )

    return results, report, predicitons , labels





In [76]:
#test evaluate

results, report, predicitons, labels = evaluate(test_model, test_loader, tensors["label_encoder"])

print(results)
print()
print(report)

{'accuracy': 0.8491561181434599, 'micro_precision': 0.8491561181434599, 'micro_recall': 0.8491561181434599, 'micro_f1': 0.8491561181434599, 'macro_precision': 0.8548555045849534, 'macro_recall': 0.8378787878787879, 'macro_f1': 0.8419788888713811}

               precision    recall  f1-score   support

     business       0.75      0.80      0.77       160
entertainment       0.77      0.89      0.83       150
       health       0.94      0.85      0.89       150
     politics       0.87      0.86      0.87       165
       sports       0.89      0.96      0.93       200
   technology       0.90      0.67      0.77       123

     accuracy                           0.85       948
    macro avg       0.85      0.84      0.84       948
 weighted avg       0.85      0.85      0.85       948



Feature Extraction

In [81]:
#func to centrally fun full training experiment 

def run_training(lang_data, lang_name,method, max_features, min_freq, lr, batch_size, num_epochs = 50, patience =3):

    #vectorise

    vec_output= vectorise_splits(lang_data, method=method, max_features=max_features, min_freq=min_freq)

    #prepare data
    tensors = prepare_tensors(lang_data, vec_output)

    #dataloaders

    train_loader, dev_loader, test_loader = build_dataloaders(tensors, batch_size)

    #create model
    input_size= tensors["x_train"].shape[1]
    num_classes = len(tensors["label_encoder"].classes_)

    model = MultinomialLogisticRegression(input_size, num_classes)

    history = train_model(model, train_loader,dev_loader, lr=lr, num_epochs=num_epochs, patience=patience)

    dev_results, report, predictions, labels = evaluate(
    model, dev_loader, tensors["label_encoder"]

)



    return {
        "language": lang_name,
        "method": method,
        "max_features": max_features ,
        "min_freq" :  min_freq ,
        "lr":  lr,
        "batch_size":batch_size,
        "vocab_size": input_size,
        "epochs_run":  len(history["train_loss"]),
        "dev_accuracy":dev_results["accuracy"],
        "dev_macro_f1": dev_results["macro_f1"],
        "model": model ,
        "history" : history,
        "tensors": tensors
      }



    


In [ ]:
results_log = []

for method in ["cvec", "tfidf"]:
    for min_freq in [1, 3, 5, 10]:

        result = run_training(
            eng,
            "english",
            method,
            5000,
            min_freq,
            0.1,
            32
        )

        results_log.append(result)

        print(
            method,
            min_freq,
            result["vocab_size"],
            result["dev_accuracy"]
        )
    #note that skilearn mindf cant be 0. therefore im choosing 

Epoch 1/50 Train loss:24.8517 Dev loss: 15.4870 Dev Accuracy:0.4873
Epoch 2/50 Train loss:7.0058 Dev loss: 2.5718 Dev Accuracy:0.7945
Epoch 3/50 Train loss:4.7041 Dev loss: 6.0491 Dev Accuracy:0.6801
Epoch 4/50 Train loss:2.9595 Dev loss: 8.1733 Dev Accuracy:0.6462
Epoch 5/50 Train loss:1.8673 Dev loss: 2.2937 Dev Accuracy:0.8326
Epoch 6/50 Train loss:1.1496 Dev loss: 1.4608 Dev Accuracy:0.8644
Epoch 7/50 Train loss:0.8630 Dev loss: 1.7238 Dev Accuracy:0.8453
Epoch 8/50 Train loss:1.7780 Dev loss: 1.6511 Dev Accuracy:0.8665
Epoch 9/50 Train loss:0.6104 Dev loss: 1.3270 Dev Accuracy:0.8665
Epoch 10/50 Train loss:0.3814 Dev loss: 1.5321 Dev Accuracy:0.8729
Epoch 11/50 Train loss:0.3433 Dev loss: 1.3425 Dev Accuracy:0.8644
Epoch 12/50 Train loss:0.1675 Dev loss: 7.6202 Dev Accuracy:0.6758
Epoch 13/50 Train loss:0.5880 Dev loss: 1.3654 Dev Accuracy:0.8559
early stop
cvec 1 5000 0.8559322033898306
Epoch 1/50 Train loss:26.9846 Dev loss: 72.3257 Dev Accuracy:0.3686
Epoch 2/50 Train loss:7.77